In [6]:
from copy import deepcopy

def remap_coco_category_ids(
    coco_data: dict,
    start_at: int = 0,
    order: str = "categories_order",  # "categories_order" or "sorted_ids"
    preserve_original_ids: bool = True,
    drop_unknown_annots: bool = False
):
    """
    Remap category IDs in a COCO dataset dictionary to be contiguous starting at `start_at`.
    Updates both `categories` and `annotations`.

    Parameters
    ----------
    coco_data : dict
        A COCO-style dict with keys: 'images', 'annotations', 'categories', etc.
    start_at : int
        Starting ID for remapped classes (default: 0).
    order : str
        - "categories_order": preserve the order of coco_data['categories']
        - "sorted_ids": sort by the original numeric category id
    preserve_original_ids : bool
        If True, attach 'orig_id' to category entries and 'orig_category_id' to annotations.
    drop_unknown_annots : bool
        If True, silently drop annotations whose category_id is not in categories.
        If False, raise an error on unknown category_id.

    Returns
    -------
    remapped_data : dict
        New COCO dict with remapped category ids.
    id_map : dict[int -> int]
        Mapping from old category_id to new contiguous id.
    """
    if "categories" not in coco_data or "annotations" not in coco_data:
        raise ValueError("coco_data must contain 'categories' and 'annotations' keys.")

    categories = coco_data["categories"]
    if not categories:
        raise ValueError("No categories present to remap.")

    # Decide the ordering of categories for mapping
    if order == "sorted_ids":
        cats_ordered = sorted(categories, key=lambda c: int(c["id"]))
    elif order == "categories_order":
        cats_ordered = list(categories)
    else:
        raise ValueError("order must be 'categories_order' or 'sorted_ids'.")

    # Build old->new id mapping
    id_map = {}
    next_id = start_at
    for cat in cats_ordered:
        old_id = int(cat["id"])
        if old_id not in id_map:
            id_map[old_id] = next_id
            next_id += 1

    # Remap categories
    remapped = deepcopy(coco_data)
    new_categories = []
    for cat in cats_ordered:
        old_id = int(cat["id"])
        new_cat = deepcopy(cat)
        if preserve_original_ids:
            new_cat["orig_id"] = old_id
        new_cat["id"] = id_map[old_id]
        new_categories.append(new_cat)
    remapped["categories"] = new_categories

    # For faster membership checks
    valid_old_ids = set(id_map.keys())

    # Remap annotations
    new_annotations = []
    for ann in remapped["annotations"]:
        old_cid = int(ann["category_id"])
        if old_cid not in valid_old_ids:
            if drop_unknown_annots:
                # Skip this annotation
                continue
            else:
                raise ValueError(
                    f"Annotation refers to unknown category_id={old_cid}. "
                    f"Make sure 'categories' include all used IDs."
                )
        new_ann = deepcopy(ann)
        if preserve_original_ids:
            new_ann["orig_category_id"] = old_cid
        new_ann["category_id"] = id_map[old_cid]
        new_annotations.append(new_ann)
    remapped["annotations"] = new_annotations

    return remapped, id_map

In [7]:
import json
import os
import shutil
import numpy as np
from collections import defaultdict, Counter
from PIL import Image
from pycocotools.coco import COCO


def balanced_split_data(
    root,
    src_annotations_path='annotations/result.json',
    train_json='train.json',
    val_json='val.json',
    train_dir_name='train',
    val_dir_name='val',
    val_perc=0.20,
    seed=42,
    copy_mode='copy',
    quota_strategy='proportional',  # 'proportional' or 'equalize'
    min_instances_per_class_val=1,
    target_instances_per_class_val=None,  # used when quota_strategy='equalize'
    # ---- NEW: class filtering ----
    exclude_class_names=('mfsk', 'lsb', 'cw'),            # e.g., ['person', 'car']
    exclude_class_ids=None,              # e.g., [1, 3]
    drop_images_without_included=True    # drop images that have no remaining (included) annotations
):
    """
    Split a COCO dataset into train/val with instance-balanced validation; write annotations to JSON
    files and copy/symlink images into split-specific subdirectories. Supports excluding classes.

    Parameters
    ----------
    root : str
        Root directory of the dataset. Image paths in the COCO JSON are joined with this root.
    src_annotations_path : str
        Path to the source COCO annotations JSON, relative to root (default: 'annotations/result.json').
    train_json, val_json : str
        Output JSON filenames (written under root/annotations/).
    train_dir_name, val_dir_name : str
        Names of the image subdirectories created under root.
    val_perc : float
        Validation quota per class. For 'proportional' it's the fraction of each class's instances;
        for 'equalize' it scales an equal target per class.
    seed : int or None
        Random seed for reproducibility.
    copy_mode : str
        One of {'copy', 'move', 'symlink'} controlling how images are placed into split dirs.
    quota_strategy : str
        'proportional' (default) – take val_perc of each class's instances.
        'equalize' – aim for near-equal instances per class (uses target_instances_per_class_val if provided).
    min_instances_per_class_val : int
        Minimum instances per class to place in val (if instances exist in the dataset).
    target_instances_per_class_val : int or None
        When using 'equalize', desired per-class instance count in val; if None, auto-estimated.

    exclude_class_names : list[str] or None
        Class names to exclude (case-insensitive). Names must match COCO category 'name' field.
    exclude_class_ids : list[int] or None
        Category IDs to exclude.
    drop_images_without_included : bool
        If True (default), drop images that have zero remaining (included) annotations after filtering.

    Returns
    -------
    dict
        Paths to created directories and JSON filenames (relative names, matching your original).
    """
    if seed is not None:
        np.random.seed(seed)

    coco_path = os.path.join(root, src_annotations_path)
    coco_ds = COCO(coco_path)

    # ---- Build excluded ID set from names and/or ids ----
    name_to_ids = defaultdict(list)
    for cid, cat in coco_ds.cats.items():
        name_to_ids[cat['name'].strip().lower()].append(cid)

    excluded_ids = set()
    if exclude_class_names:
        for nm in exclude_class_names:
            key = nm.strip().lower()
            if key in name_to_ids:
                excluded_ids.update(name_to_ids[key])
            else:
                print(f"WARNING: exclude_class_names entry '{nm}' not found in categories; skipping.")
    if exclude_class_ids:
        excluded_ids.update(int(x) for x in exclude_class_ids)

    # IDs to include and filtered categories list
    categories_included = [cid for cid in coco_ds.cats.keys() if cid not in excluded_ids]
    categories_filtered = [coco_ds.cats[cid] for cid in categories_included]

    if len(categories_included) == 0:
        raise ValueError("After filtering, no categories remain. "
                         "Check exclude_class_names/exclude_class_ids.")

    # Collect candidate images: file exists, not corrupt, and has included annotations
    img_ids = coco_ds.getImgIds()
    np.random.shuffle(img_ids)

    valid_ids = []
    img_info_map = {}
    img_anns_map = {}
    img_cat_counts = {}  # img_id -> Counter({cat_id: count})

    for img_id in img_ids:
        ann_ids = coco_ds.getAnnIds(imgIds=[img_id], iscrowd=None)
        if len(ann_ids) == 0:
            continue

        anns = coco_ds.loadAnns(ann_ids)
        # Filter out excluded classes from annotations
        included_anns = [a for a in anns if a['category_id'] in categories_included]

        # Optionally drop images that have no included annotations left
        if drop_images_without_included and len(included_anns) == 0:
            continue

        img_info = coco_ds.imgs[img_id]
        src_path = os.path.join(root, img_info['file_name'])
        if not os.path.isfile(src_path):
            print(f"Missing file: {src_path}. Skipping image id {img_id}.")
            continue

        # Verify image isn't corrupt
        try:
            with Image.open(src_path) as im:
                im.verify()
        except Exception as e:
            print(f"Corrupt image {img_info['file_name']} (id {img_id}), skipping: {e}")
            continue

        # Build per-image category instance counts (only included categories)
        counts = Counter()
        for a in included_anns:
            counts[a['category_id']] += 1

        # If we kept images with zero included annotations (drop_images_without_included=False),
        # skip them anyway for splitting (they don't contribute to quotas)
        if sum(counts.values()) == 0:
            continue

        valid_ids.append(img_id)
        img_info_map[img_id] = img_info
        img_anns_map[img_id] = included_anns
        img_cat_counts[img_id] = counts

    # Compute total instances per included category across valid images
    total_per_cat = {c: 0 for c in categories_included}
    for img_id in valid_ids:
        for c, k in img_cat_counts[img_id].items():
            total_per_cat[c] += k

    number_of_imgs = len(valid_ids)
    print(f"""
    Total valid images (after filtering): {number_of_imgs}
    Included classes:
      {[coco_ds.cats[c]['name'] for c in categories_included]}
    Total instances per included class:
      {{ {', '.join([f"{coco_ds.cats[c]['name']}: {total_per_cat[c]}" for c in categories_included])} }}
    """)

    # --- Quotas computation for VAL ---
    def compute_val_quotas(totals, perc, min_per_class, target_instances_per_class, strategy):
        quotas = {}
        if strategy == 'proportional':
            for c in categories_included:
                t = totals[c]
                q = int(np.round(t * perc)) if t > 0 else 0
                q = min(max(min_per_class if t > 0 else 0, q), t)
                quotas[c] = q
        elif strategy == 'equalize':
            nonzero = [totals[c] for c in categories_included if totals[c] > 0]
            if target_instances_per_class is None:
                base = int(np.round(np.mean(nonzero) * perc)) if nonzero else 0
                target = max(min_per_class, base)
            else:
                target = target_instances_per_class
            for c in categories_included:
                t = totals[c]
                q = min(target, t)
                q = max(min_per_class if t > 0 else 0, q)
                quotas[c] = q
        else:
            raise ValueError("quota_strategy must be 'proportional' or 'equalize'")
        return quotas

    # Greedy selection: choose images that reduce unmet quotas the most
    def select_images_for_val(pool_ids, quotas, counts_map):
        selected = []
        quotas = quotas.copy()
        pool_set = set(pool_ids)
        while any(quotas[c] > 0 for c in categories_included):
            best_id, best_gain = None, 0
            for img_id in list(pool_set):
                counts = counts_map[img_id]
                gain = sum(min(quotas[c], counts.get(c, 0)) for c in categories_included)
                if gain > best_gain:
                    best_gain = gain
                    best_id = img_id
            if best_gain == 0 or best_id is None:
                break
            selected.append(best_id)
            for c, k in counts_map[best_id].items():
                if quotas.get(c, 0) > 0:
                    quotas[c] = max(0, quotas[c] - k)
            pool_set.remove(best_id)
        return selected, quotas, list(pool_set)

    # Prepare output dirs
    split_dirs = {
        'train': os.path.join(root, train_dir_name),
        'val': os.path.join(root, val_dir_name),
    }
    for p in split_dirs.values():
        os.makedirs(p, exist_ok=True)

    def place_image(src_path, dst_path, mode='copy'):
        os.makedirs(os.path.dirname(dst_path), exist_ok=True)
        if mode == 'move':
            shutil.move(src_path, dst_path)
        elif mode == 'symlink':
            try:
                if os.path.lexists(dst_path):
                    os.remove(dst_path)
                os.symlink(os.path.abspath(src_path), dst_path)
            except Exception:
                shutil.copy2(src_path, dst_path)
        else:
            shutil.copy2(src_path, dst_path)

    # --- Build VAL split ---
    val_quotas = compute_val_quotas(
        totals=total_per_cat,
        perc=val_perc,
        min_per_class=min_instances_per_class_val,
        target_instances_per_class=target_instances_per_class_val,
        strategy=quota_strategy
    )
    val_selected, val_unmet, remaining_after_val = select_images_for_val(valid_ids, val_quotas, img_cat_counts)

    # TRAIN is everything else
    train_selected = remaining_after_val

    # Assemble COCO dicts (with filtered categories)
    def new_split_dict():
        d = {
            'images': [],
            'annotations': [],
            'categories': categories_filtered
        }
        if 'licenses' in coco_ds.dataset:
            d['licenses'] = coco_ds.dataset['licenses']
        if 'info' in coco_ds.dataset:
            d['info'] = coco_ds.dataset['info']
        d['type'] = coco_ds.dataset.get('type', 'instances')
        return d

    train_dict = new_split_dict()
    val_dict = new_split_dict()

    def add_img_and_anns(img_id, target_dict, split_key):
        img_info = img_info_map[img_id]
        file_name = img_info['file_name']
        src_path = os.path.join(root, file_name)
        target_dict['images'].append(img_info)
        target_dict['annotations'].extend(img_anns_map[img_id])
        dst_path = os.path.join(split_dirs[split_key], file_name)
        place_image(src_path, dst_path, mode=copy_mode)

    for img_id in val_selected:
        add_img_and_anns(img_id, val_dict, 'val')
    for img_id in train_selected:
        add_img_and_anns(img_id, train_dict, 'train')

    # Remap category IDs for YOLOX compatibility (contiguous starting at 0)
    train_dict_remap, train_idmap = remap_coco_category_ids(train_dict, start_at=0, order="categories_order")
    val_dict_remap,   val_idmap   = remap_coco_category_ids(val_dict,   start_at=0, order="categories_order")

    # Optional: sanity check that both splits use the same mapping (same category set/order)
    assert train_idmap == val_idmap, "Train/Val category sets differ; ensure categories are identical across splits."

    # Write JSONs to root/annotations/<name>.json
    def write_json(path, data):
        os.makedirs(os.path.dirname(path), exist_ok=True)
        with open(path, 'w', encoding='utf-8') as f:
            json.dump(data, f, ensure_ascii=False, indent=2)

    train_json_path = os.path.join(root, "annotations", train_json)
    val_json_path = os.path.join(root, "annotations", val_json)

    # Then write the remapped dicts
    write_json(train_json_path, train_dict_remap)
    write_json(val_json_path, val_dict_remap)

    # Report per-class instance counts in each split (included categories only)
    def per_class_counts(split_dict):
        pc = defaultdict(int)
        for a in split_dict['annotations']:
            if a['category_id'] in categories_included:
                pc[a['category_id']] += 1
        return {coco_ds.cats[c]['name']: pc[c] for c in categories_included}

    val_counts = per_class_counts(val_dict)
    train_counts = per_class_counts(train_dict)

    planned_val_total = sum(val_quotas.values())
    actual_val_total = sum(val_counts.values())

    print(f"""
    Planned VAL instances total: {planned_val_total}
    Actual  VAL instances total: {actual_val_total}

    Final split (after filtering):
      Train images: {len(train_dict['images'])}
      Val   images: {len(val_dict['images'])}
    Per-class instance counts (included classes):
      Val:  {val_counts}
      Train:{train_counts}

    Unmet VAL quotas (remaining after selection):
      {{ {', '.join([f"{coco_ds.cats[c]['name']}: {val_unmet[c]}" for c in categories_included])} }}

    Image output directories:
      Train -> {split_dirs['train']}
      Val   -> {split_dirs['val']}

    JSON files:
      Train -> {train_json_path}
      Val   -> {val_json_path}
    """)

    # Warn if any included class has instances overall but none in VAL
    missing_val = [coco_ds.cats[c]['name'] for c in categories_included
                   if total_per_cat[c] > 0 and val_counts.get(coco_ds.cats[c]['name'], 0) == 0]
    if missing_val:
        print(f"WARNING: No instances in VAL for these included classes (despite availability): {missing_val}")

    # Inform about exclusions
    if excluded_ids:
        excluded_names = sorted({coco_ds.cats[c]['name'] for c in excluded_ids if c in coco_ds.cats})
        print(f"Excluded classes: {excluded_names}")

    return {
        'train_dir': split_dirs['train'],
        'val_dir': split_dirs['val'],
        'train_json': train_json,  # keeping your original return style (relative names)
        'val_json': val_json
    }


In [8]:
root = "/workspaces/hfml/dataset"
balanced_split_data(root)


loading annotations into memory...
Done (t=0.02s)
creating index...
index created!

    Total valid images (after filtering): 1513
    Included classes:
      ['broadcast_am', 'china_oth_160', 'ft', 'hfdl', 'ofdm', 'rtty', 'stanag_wbhf', 'usb']
    Total instances per included class:
      { broadcast_am: 313, china_oth_160: 264, ft: 250, hfdl: 265, ofdm: 283, rtty: 257, stanag_wbhf: 260, usb: 434 }
    

    Planned VAL instances total: 466
    Actual  VAL instances total: 472

    Final split (after filtering):
      Train images: 1348
      Val   images: 165
    Per-class instance counts (included classes):
      Val:  {'broadcast_am': 63, 'china_oth_160': 55, 'ft': 50, 'hfdl': 53, 'ofdm': 57, 'rtty': 51, 'stanag_wbhf': 52, 'usb': 91}
      Train:{'broadcast_am': 250, 'china_oth_160': 209, 'ft': 200, 'hfdl': 212, 'ofdm': 226, 'rtty': 206, 'stanag_wbhf': 208, 'usb': 343}

    Unmet VAL quotas (remaining after selection):
      { broadcast_am: 0, china_oth_160: 0, ft: 0, hfdl: 0, ofdm

{'train_dir': '/workspaces/hfml/dataset/train',
 'val_dir': '/workspaces/hfml/dataset/val',
 'train_json': 'train.json',
 'val_json': 'val.json'}